In [2]:
import os
import cv2
import albumentations as A
import subprocess

# Video Format Conversion and Frame Extraction

In [29]:
input_video_path = ('/home/vuminhdangle/Documents/VuMinhDangLE/Realtime-Object-Counting/Video/input_video.webm')

out_dir = ('/home/vuminhdangle/Documents/VuMinhDangLE/Realtime-Object-Counting/Video')

mp4_dir = os.path.join(out_dir, 'mp4')
frames_dir = os.path.join(out_dir, 'frames')

os.makedirs(mp4_dir, exist_ok=True)
os.makedirs(frames_dir, exist_ok=True)

In [33]:
if not os.path.isfile(input_video_path):
    raise FileNotFoundError(
        f"Input video not found: {input_video_path}"
    )

# KEEP THE ORIGINAL FILENAME AND CHANGE THE EXTENSION
video_name = os.path.splitext(
    os.path.basename(input_video_path)
)[0]

mp4_path = os.path.join(mp4_dir, video_name + '.mp4')
# video_frames_dir = os.path.join(frames_dir, video_name)
video_frames_dir = frames_dir

os.makedirs(video_frames_dir, exist_ok=True)

# CONVERT WEBM TO MP4

command = [
    'ffmpeg',
    '-y',
    '-i', input_video_path,
    '-c:v', 'libx264',
    '-crf', '18',
    '-preset', 'medium',
    '-c:a', 'aac',
    '-b:a', '192k',
    mp4_path
]

subprocess.run(command, check=True)

print(f'MP4 saved to: {mp4_path}')

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

MP4 saved to: /home/vuminhdangle/Documents/VuMinhDangLE/Realtime-Object-Counting/Video/mp4/input_video.mp4


frame=  840 fps=404 q=-1.0 Lsize=    9817kB time=00:00:27.99 bitrate=2873.0kbits/s speed=13.5x    
video:9778kB audio:8kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 0.322132%
[libx264 @ 0x595217760940] frame I:4     Avg QP:16.61  size:109622
[libx264 @ 0x595217760940] frame P:212   Avg QP:19.14  size: 30314
[libx264 @ 0x595217760940] frame B:624   Avg QP:24.95  size:  5043
[libx264 @ 0x595217760940] consecutive B-frames:  0.8%  0.2%  0.4% 98.6%
[libx264 @ 0x595217760940] mb I  I16..4:  2.1% 43.6% 54.3%
[libx264 @ 0x595217760940] mb P  I16..4:  0.4%  6.3%  3.5%  P16..4: 35.3% 28.0% 20.1%  0.0%  0.0%    skip: 6.4%
[libx264 @ 0x595217760940] mb B  I16..4:  0.0%  0.4%  0.3%  B16..8: 43.5%  8.6%  3.2%  direct: 2.8%  skip:41.3%  L0:54.3% L1:29.8% BI:15.9%
[libx264 @ 0x595217760940] 8x8 transform intra:58.7% inter:61.0%
[libx264 @ 0x595217760940] coded y,uvDC,uvAC intra: 89.1% 88.0% 59.6% inter: 22.2% 17.3% 4.7%
[libx264 @ 0x595217760940] i16 v,h,dc,p: 21% 17%  9% 53%


In [34]:
# EXTRACT ALL FRAMES

cap = cv2.VideoCapture(input_video_path)

if not cap.isOpened():
    raise RuntimeError(f'Cannot open video: {input_video_path}')

frame_idx = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    frame_path = os.path.join(
        video_frames_dir,
        f'frame_{frame_idx}.png'
    )

    if not cv2.imwrite(frame_path, frame):
        cap.release()
        raise RuntimeError(f'Failed to save frame: {frame_path}')

    frame_idx += 1

cap.release()

print(f'Extracted {frame_idx} frames.')
print(f'Frames saved to: {video_frames_dir}')
print('Processing completed successfully!')

Extracted 840 frames.
Frames saved to: /home/vuminhdangle/Documents/VuMinhDangLE/Realtime-Object-Counting/Video/frames
Processing completed successfully!


# DATA AUGMENTATION

In [3]:
# INPUTS
image_path = "/home/vuminhdangle/Documents/VuMinhDangLE/Realtime-Object-Counting/dataset/train/images/frame_0.jpg"
label_path = "/home/vuminhdangle/Documents/VuMinhDangLE/Realtime-Object-Counting/dataset/train/labels/frame_0.txt"
output_dir_images = '/home/vuminhdangle/Documents/VuMinhDangLE/Realtime-Object-Counting/dataset/train/images/'
output_dir_labels = '/home/vuminhdangle/Documents/VuMinhDangLE/Realtime-Object-Counting/dataset/train/labels/'
num_images = 499 # Generate 500 images

os.makedirs(output_dir_images, exist_ok=True)
os.makedirs(output_dir_labels, exist_ok=True)

In [4]:
# LOAD ORIGINAL IMAGE
image = cv2.imread(image_path)
h, w = image.shape[:2]

# LOAD YOLO ANNOTATION
with open(label_path, 'r') as f:
    yolo_data = f.readlines()

bboxes = []
classes = []

for line in yolo_data:
    cls, x, y, bw, bh = map(float, line.split())
    classes.append(int(cls))
    bboxes.append([x, y, bw, bh])

In [5]:
# AUGMENTATION PIPELINE
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=20, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.3),
    A.MotionBlur(p=0.3),
    A.HueSaturationValue(p=0.5),
    A.RandomGamma(p=0.5),
    A.Perspective(p=0.5),
    A.RandomScale(scale_limit=0.2, p=0.5),
    A.Affine(translate_percent=(-0.05,0.05), scale=(0.8,1.2), rotate=(-15, 15), p=0.7),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.2, rotate_limit=15, p=0.7)
], bbox_params=A.BboxParams(format='yolo', label_fields=['category_ids']))

# GEMERATE IMAGES

for i in range(num_images):
    print(f"\rGenerating image {i + 1}/{num_images}", end="", flush=True)

    augmented = transform(image=image, bboxes=bboxes, category_ids=classes)
    aug_img = augmented["image"]
    aug_bboxes = augmented["bboxes"]

    img_name = f"frame_{i+1}.jpg"
    txt_name = f"frame_{i+1}.txt"

    cv2.imwrite(os.path.join(output_dir_images, img_name), aug_img)

    # SAVE UPDATED YOLO LABEL
    with open(os.path.join(output_dir_labels, txt_name), 'w') as f:
        for cls, box in zip(classes, aug_bboxes):
            f.write(f"{cls} {' '.join(map(str, box))}\n")

print(f"\r{num_images}/{num_images} images generated successfully!")

Generating image 28/499

/home/vuminhdangle/Documents/VuMinhDangLE/Realtime-Object-Counting/.venv/lib/python3.10/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


499/499 images generated successfully!
